# 05.01 — Training Dataset

## Purpose

Construct and inspect the event-level modeling table for
$P(\mathrm{rating} \ge 4 \mid \text{information available before } t)$.

The join is an identity-safe indexed join: Phase 2 defines `ratingEventId` as the
1-based canonical source-row identity, so labels are selected from canonical ratings
by that key. IDs, timestamps, outcomes, and provenance never enter `FEATURE_COLUMNS`.

## Setup and explicit predictor contract

The repository root is discovered from stable markers, so paths work from any directory inside the repository.
Only required columns are read from the 20M-row artifacts.

In [1]:
from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CURRENT_PATH = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (CURRENT_PATH, *CURRENT_PATH.parents)
     if (path / "README.md").is_file() and (path / ".git").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the movie_rating_predictor repository root")

FEATURE_PATH = PROJECT_ROOT / "data" / "features" / "rating_features_v1.parquet"
FEATURE_METADATA_PATH = PROJECT_ROOT / "data" / "features" / "rating_features_v1.metadata.json"
RATINGS_PATH = PROJECT_ROOT / "data" / "processed" / "ratings.parquet"
MODELS_DIR = PROJECT_ROOT / "models"

FEATURE_COLUMNS = [
    "global_rating_count",
    "global_mean_rating",
    "user_rating_count",
    "user_mean_rating",
    "user_rating_std_pop",
    "user_seconds_since_last_rating",
    "movie_rating_count",
    "movie_mean_rating",
    "movie_rating_std_pop",
    "movie_rating_count_30d",
    "user_target_genre_rating_count",
    "user_target_genre_mean_rating",
    "user_target_genre_mean_delta",
    "movie_genre_count",
    "movie_release_year",
    "movie_release_year_missing",
    "movie_age_years",
]

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Random seed:", RANDOM_SEED)
print("Feature count:", len(FEATURE_COLUMNS))
print("Features:", FEATURE_COLUMNS)

Python: 3.14.6
pandas: 3.0.5
NumPy: 2.5.3
Random seed: 42
Feature count: 17
Features: ['global_rating_count', 'global_mean_rating', 'user_rating_count', 'user_mean_rating', 'user_rating_std_pop', 'user_seconds_since_last_rating', 'movie_rating_count', 'movie_mean_rating', 'movie_rating_std_pop', 'movie_rating_count_30d', 'user_target_genre_rating_count', 'user_target_genre_mean_rating', 'user_target_genre_mean_delta', 'movie_genre_count', 'movie_release_year', 'movie_release_year_missing', 'movie_age_years']


## Artifact and schema validation

Validate the generated Parquet against Phase 4 metadata before loading the modeling
columns. This does not change the Phase 4 artifact.

In [2]:
metadata = json.loads(FEATURE_METADATA_PATH.read_text())
feature_file = pq.ParquetFile(FEATURE_PATH)
ratings_file = pq.ParquetFile(RATINGS_PATH)

assert metadata["rowCount"] == feature_file.metadata.num_rows == ratings_file.metadata.num_rows
assert metadata["predictorCount"] == 17 == len(FEATURE_COLUMNS)
assert metadata["predictorNames"] == FEATURE_COLUMNS
assert set(metadata["outputSchema"]) == set(feature_file.schema_arrow.names)

print("Rows:", f'{metadata["rowCount"]:,}')
print("Feature contract:", metadata["featureContractVersion"])
print("Schema version:", metadata["materializationSchemaVersion"])
print("History source:", metadata["historySourceId"])
print("Catalog snapshot:", metadata["catalogSnapshotId"])

Rows: 20,000,263
Feature contract: FEATURE_DICTIONARY_V1
Schema version: 1
History source: 283ca831aba8fb4b3abc18cd369f810169085453a25ef0971f80a012a20e206e
Catalog snapshot: 2f0f00445304627872fa507bb2759c4a81744ff91c24901d426f95f6941cadb8


## Load and join labels

The feature frame contains context plus exactly 17 predictors. Canonical ratings
contribute only `rating` and the independent timestamp used to audit the identity
join. Chunked timestamp checks avoid creating another full 20M-row timestamp copy.

In [3]:
feature_columns_to_read = ["ratingEventId", "userId", "movieId", "timestamp", *FEATURE_COLUMNS]
features = pd.read_parquet(FEATURE_PATH, columns=feature_columns_to_read)
ratings = pd.read_parquet(RATINGS_PATH, columns=["rating", "timestamp"])

event_ids = features["ratingEventId"].to_numpy(dtype=np.uint64, copy=False)
expected_rows = len(ratings)
assert len(features) == expected_rows
assert event_ids.min() == 1 and event_ids.max() == expected_rows
assert pd.Series(event_ids, copy=False).is_unique

rating_values = ratings["rating"].to_numpy(dtype=np.float32, copy=False)
y = (rating_values[event_ids - 1] >= 4.0).astype(np.int8)

chunk_size = 1_000_000
rating_timestamps = ratings["timestamp"].to_numpy(copy=False)
feature_timestamps = features["timestamp"].to_numpy(copy=False)
for start in range(0, expected_rows, chunk_size):
    stop = min(start + chunk_size, expected_rows)
    np.testing.assert_array_equal(
        feature_timestamps[start:stop],
        rating_timestamps[event_ids[start:stop] - 1],
    )

modeling_contract = features[["ratingEventId", "timestamp", *FEATURE_COLUMNS]]
print("Identity join and timestamp audit: PASS")
print("Modeling contract shape:", modeling_contract.shape)
print("X shape:", (len(features), len(FEATURE_COLUMNS)), "y shape:", y.shape)

Identity join and timestamp audit: PASS
Modeling contract shape: (20000263, 19)
X shape: (20000263, 17) y shape: (20000263,)


## Modeling-table summary

The original timestamp is retained only for splitting. `highRating` is the label,
not a predictor.

In [4]:
summary = pd.Series({
    "observations": len(features),
    "positive_rows": int(y.sum()),
    "negative_rows": int((y == 0).sum()),
    "target_prevalence": float(y.mean()),
    "min_timestamp": features["timestamp"].min(),
    "max_timestamp": features["timestamp"].max(),
    "feature_count": len(FEATURE_COLUMNS),
})
display(summary.to_frame("value"))

class_counts = pd.Series(y, name="highRating").value_counts().sort_index().rename_axis("highRating")
display(class_counts.to_frame("rows"))

,value
observations,20000263
positive_rows,9995410
negative_rows,10004853
target_prevalence,0.499764
min_timestamp,1995-01-09 11:46:44
max_timestamp,2015-03-31 06:40:02
feature_count,17


,rows
highRating,
0,10004853
1,9995410


## Dtypes, missingness, and descriptive statistics

Missing recency and release-year values are intentional. XGBoost will use its native
missing-value routing; no future-derived imputation is introduced.

In [5]:
dtype_and_missingness = pd.DataFrame({
    "dtype": features[FEATURE_COLUMNS].dtypes.astype(str),
    "missing_rows": features[FEATURE_COLUMNS].isna().sum(),
})
dtype_and_missingness["missing_pct"] = 100 * dtype_and_missingness["missing_rows"] / len(features)
display(dtype_and_missingness)

descriptive_statistics = features[FEATURE_COLUMNS].describe(percentiles=[.01, .1, .5, .9, .99]).T
display(descriptive_statistics)

,dtype,missing_rows,missing_pct
global_rating_count,uint64,0,0.000000
global_mean_rating,float32,0,0.000000
user_rating_count,uint64,0,0.000000
user_mean_rating,float32,0,0.000000
user_rating_std_pop,float32,0,0.000000
user_seconds_since_last_rating,float64,274729,1.373627
movie_rating_count,uint64,0,0.000000
movie_mean_rating,float32,0,0.000000
movie_rating_std_pop,float32,0,0.000000
movie_rating_count_30d,uint64,0,0.000000


,count,mean,std,min,1%,10%,50%,90%,99%,max
global_rating_count,20000263.0,10000130.485351,5773579.266339,0.0,200002.0,2000025.4,10000131.0,18000235.8,19800259.38,20000262.0
global_mean_rating,20000263.0,3.540222,0.033326,3.5,3.510953,3.512632,3.532335,3.574231,3.594455,4.18
user_rating_count,20000263.0,254.831463,410.582294,0.0,0.0,13.0,115.0,640.0,1942.0,9253.0
user_mean_rating,20000263.0,3.553494,0.468647,0.5,2.227273,2.978967,3.578592,4.107143,4.521739,5.0
user_rating_std_pop,20000263.0,0.910692,0.269474,0.0,0.0,0.622512,0.910187,1.230978,1.549317,2.25
user_seconds_since_last_rating,19725534.0,132409.288922,2209457.491028,1.0,1.0,4.0,24.0,186.0,2080240.83,464596991.0
movie_rating_count,20000263.0,6739.69127,9029.228089,0.0,10.0,214.0,3223.0,18285.0,42728.38,67309.0
movie_mean_rating,20000263.0,3.574215,0.487685,0.5,2.101562,2.920816,3.653061,4.13408,4.40774,5.0
movie_rating_std_pop,20000263.0,0.930043,0.130212,0.0,0.650786,0.78878,0.924322,1.0879,1.246399,2.25
movie_rating_count_30d,20000263.0,181.03281,385.339482,0.0,0.0,7.0,63.0,381.0,2215.0,3724.0


## Leakage guard

The allow-list is authoritative. This assertion prevents accidental context, identity,
target, or provenance columns from becoming model inputs.

In [6]:
FORBIDDEN_PREDICTORS = {
    "ratingEventId", "userId", "movieId", "timestamp", "rating", "highRating",
    "historySourceId", "catalogSnapshotId",
}
assert len(FEATURE_COLUMNS) == 17
assert not (set(FEATURE_COLUMNS) & FORBIDDEN_PREDICTORS)
assert list(metadata["predictorNames"]) == FEATURE_COLUMNS
print("Leakage allow-list check: PASS")

Leakage allow-list check: PASS
